<img src="https://res.cloudinary.com/dtizipxds/image/upload/q_auto/f_auto/v1776264397/banner_yvwajv.png" width="100%">


In [ ]:
%pip install numpy pandas matplotlib seaborn scikit-learn


# Template - Feature Engineering
This notebook shows many practical feature engineering techniques with simple code.

Techniques covered:
- Missing value handling
- Outlier capping
- Encoding categorical variables
- Scaling and transforms
- Binning
- Interaction and polynomial features
- Date/time features
- Text-derived features
- Group-based aggregate features
- Basic feature selection


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, RobustScaler, PolynomialFeatures
from sklearn.feature_selection import mutual_info_classif

sns.set_theme(style="whitegrid")
np.random.seed(42)


## Part A) Numeric + Categorical Feature Engineering (Wine dataset)


In [ ]:
wine = load_wine(as_frame=True)
df = wine.frame.copy()
df = df.rename(columns={"target": "target_class"})

# Add a categorical feature for demonstration.
regions = np.array(["north", "south", "east"])
df["region"] = regions[np.random.randint(0, len(regions), size=len(df))]

# Add an ordinal quality label based on alcohol quantiles.
df["alcohol_band"] = pd.qcut(df["alcohol"], q=3, labels=["low", "medium", "high"])

# Inject some missing values and outliers so we can engineer features on a realistic dataset.
for col in ["alcohol", "malic_acid", "proline"]:
    miss_idx = np.random.choice(df.index, size=8, replace=False)
    df.loc[miss_idx, col] = np.nan

out_idx = np.random.choice(df.index, size=5, replace=False)
df.loc[out_idx, "proline"] = df["proline"].median() * 4

print(df.shape)
df.head()


## 1) Quick visualization before engineering


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=df, x="alcohol", kde=True, ax=axes[0], color="#3a86ff")
axes[0].set_title("Alcohol (with missing values)")

sns.boxplot(data=df, y="proline", ax=axes[1], color="#ffbe0b")
axes[1].set_title("Proline (with outliers)")

plt.tight_layout()
plt.show()


## 2) Missing-value imputation


In [ ]:
numeric_cols = ["alcohol", "malic_acid", "ash", "alcalinity_of_ash", "magnesium",
                "total_phenols", "flavanoids", "nonflavanoid_phenols", "proanthocyanins",
                "color_intensity", "hue", "od280/od315_of_diluted_wines", "proline"]
cat_cols = ["region", "alcohol_band"]

num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

df[numeric_cols] = num_imputer.fit_transform(df[numeric_cols])
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

print("Remaining missing values:", int(df.isna().sum().sum()))


## 3) Outlier capping (IQR method)


In [ ]:
def iqr_cap(series: pd.Series) -> pd.Series:
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    return series.clip(lower=low, upper=high)

for col in ["proline", "color_intensity", "alcohol"]:
    df[col] = iqr_cap(df[col])

plt.figure(figsize=(6, 4))
sns.boxplot(data=df, y="proline", color="#06d6a0")
plt.title("Proline after IQR capping")
plt.tight_layout()
plt.show()


## 4) Encoding categorical features


In [ ]:
# One-hot for nominal feature.
onehot = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
region_ohe = onehot.fit_transform(df[["region"]])
region_cols = [f"region_{c}" for c in onehot.categories_[0]]
region_df = pd.DataFrame(region_ohe, columns=region_cols, index=df.index)

# Ordinal encoding for ordered label.
ordinal = OrdinalEncoder(categories=[["low", "medium", "high"]])
df["alcohol_band_ord"] = ordinal.fit_transform(df[["alcohol_band"]])

engineered = pd.concat([df.drop(columns=["region", "alcohol_band"]), region_df], axis=1)
engineered.head()


## 5) Scaling and transforms


In [ ]:
# Log transform skewed variables and robust-scale a selected numeric block.
for col in ["proline", "color_intensity"]:
    engineered[f"log_{col}"] = np.log1p(engineered[col])

scale_cols = ["alcohol", "malic_acid", "proline", "color_intensity"]
scaler = RobustScaler()
engineered[[f"scaled_{c}" for c in scale_cols]] = scaler.fit_transform(engineered[scale_cols])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(engineered["proline"], kde=True, ax=axes[0], color="#ef476f")
axes[0].set_title("Proline (raw)")
sns.histplot(engineered["log_proline"], kde=True, ax=axes[1], color="#118ab2")
axes[1].set_title("Proline (log1p)")
plt.tight_layout()
plt.show()


## 6) Binning + interaction features


In [ ]:
# Binning turns continuous variables into grouped categories.
engineered["alcohol_bin"] = pd.qcut(engineered["alcohol"], q=4, labels=["q1", "q2", "q3", "q4"])

# Simple interactions / ratios.
engineered["phenols_x_flavanoids"] = engineered["total_phenols"] * engineered["flavanoids"]
engineered["malic_to_alcohol"] = engineered["malic_acid"] / (engineered["alcohol"] + 1e-6)

engineered[["alcohol", "alcohol_bin", "phenols_x_flavanoids", "malic_to_alcohol"]].head()


## 7) Polynomial features (small subset)


In [ ]:
poly_input_cols = ["alcohol", "malic_acid", "hue"]
poly = PolynomialFeatures(degree=2, include_bias=False)
poly_array = poly.fit_transform(engineered[poly_input_cols])
poly_cols = poly.get_feature_names_out(poly_input_cols)
poly_df = pd.DataFrame(poly_array, columns=[f"poly_{c}" for c in poly_cols], index=engineered.index)

engineered = pd.concat([engineered, poly_df], axis=1)
print("Engineered shape after polynomial features:", engineered.shape)


## 8) Feature selection preview (mutual information)


In [ ]:
X_for_mi = engineered.select_dtypes(include=["number"]).drop(columns=["target_class"])
y = engineered["target_class"]
mi = mutual_info_classif(X_for_mi, y, random_state=42)

mi_df = pd.DataFrame({"feature": X_for_mi.columns, "mi_score": mi}).sort_values("mi_score", ascending=False)
mi_df.head(12)


In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=mi_df.head(10), x="mi_score", y="feature", color="#3a86ff")
plt.title("Top engineered features by mutual information")
plt.tight_layout()
plt.show()


## Part B) Date/Time + Text + Group Aggregations (synthetic orders dataset)


In [ ]:
n = 500
cities = np.array(["Algiers", "Oran", "Constantine", "Annaba", "Tlemcen"])
products = np.array([
    "laptop stand",
    "gaming mouse",
    "mechanical keyboard",
    "usb c adapter",
    "noise cancelling headset",
    "discount office chair",
])

orders = pd.DataFrame({
    "order_id": np.arange(1, n + 1),
    "customer_id": np.random.randint(1000, 1080, size=n),
    "city": cities[np.random.randint(0, len(cities), size=n)],
    "product_title": products[np.random.randint(0, len(products), size=n)],
    "unit_price": np.random.uniform(10, 500, size=n).round(2),
    "quantity": np.random.randint(1, 6, size=n),
    "order_ts": pd.to_datetime("2026-01-01") + pd.to_timedelta(np.random.randint(0, 90 * 24 * 60, size=n), unit="m"),
})
orders["total_amount"] = orders["unit_price"] * orders["quantity"]
orders.head()


## 9) Date/time feature engineering


In [ ]:
orders["hour"] = orders["order_ts"].dt.hour
orders["day_of_week"] = orders["order_ts"].dt.dayofweek
orders["month"] = orders["order_ts"].dt.month
orders["is_weekend"] = orders["day_of_week"].isin([5, 6]).astype(int)

# Cyclical encoding for hour.
orders["hour_sin"] = np.sin(2 * np.pi * orders["hour"] / 24)
orders["hour_cos"] = np.cos(2 * np.pi * orders["hour"] / 24)

orders[["order_ts", "hour", "day_of_week", "is_weekend", "hour_sin", "hour_cos"]].head()


## 10) Text feature engineering (simple and effective)


In [ ]:
orders["title_len"] = orders["product_title"].str.len()
orders["word_count"] = orders["product_title"].str.split().str.len()
orders["has_discount_word"] = orders["product_title"].str.contains("discount", case=False).astype(int)

orders[["product_title", "title_len", "word_count", "has_discount_word"]].head()


## 11) Group-based aggregate features


In [ ]:
# Customer behavior features.
orders["cust_avg_amount"] = orders.groupby("customer_id")["total_amount"].transform("mean")
orders["cust_order_count"] = orders.groupby("customer_id")["order_id"].transform("count")

# City-level benchmark feature.
orders["city_avg_price"] = orders.groupby("city")["unit_price"].transform("mean")
orders["amount_vs_city_avg"] = orders["total_amount"] / (orders["city_avg_price"] + 1e-6)

orders[["customer_id", "city", "total_amount", "cust_avg_amount", "cust_order_count", "amount_vs_city_avg"]].head()


## 12) Visual checks on engineered order features


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=orders, x="total_amount", bins=30, ax=axes[0], color="#ef476f")
axes[0].set_title("Total amount distribution")

sns.scatterplot(data=orders, x="cust_avg_amount", y="total_amount", hue="is_weekend", alpha=0.7, ax=axes[1])
axes[1].set_title("Customer average vs order total")
plt.tight_layout()
plt.show()


## Final note
Use this template as a toolbox: pick only the techniques that match your dataset and objective.
